In [20]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("../../data/processed/bikeshare.db")

In [2]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_dep ON hourly_departures (Start_Station_Id, hour)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_arr ON hourly_arrivals (End_Station_Id, hour)")
conn.commit()

In [3]:
conn.execute("DROP TABLE IF EXISTS station_hourly_demand")
conn.execute("""
    CREATE TABLE station_hourly_demand AS
    SELECT
        COALESCE(d.Start_Station_Id, a.End_Station_Id) AS station_id,
        COALESCE(d.hour, a.hour)                       AS hour,
        COALESCE(d.departures, 0)                      AS departures,
        COALESCE(a.arrivals, 0)                        AS arrivals
    FROM hourly_departures AS d
    FULL OUTER JOIN hourly_arrivals AS a
        ON d.Start_Station_Id = a.End_Station_Id
       AND d.hour = a.hour
""")
conn.commit()

In [5]:
pd.read_sql("SELECT COUNT(*) FROM station_hourly_demand", conn)


,COUNT(*)
0,11790751


In [6]:
pd.read_sql("SELECT * FROM station_hourly_demand LIMIT 5", conn)

,station_id,hour,departures,arrivals
0,7000,2023-01-01-00:00,2,0
1,7000,2023-01-01-01:00,5,4
2,7000,2023-01-01-02:00,1,6
3,7000,2023-01-01-04:00,1,0
4,7000,2023-01-01-08:00,1,1


In [18]:
conn.close()